In [1]:
import pandas as pd
import scanpy as sc

In [2]:
BASE_DIR = 'D:/Gdrive/notebook/2509_scrnaseq_deconv/260123_puseudo_data'

In [3]:
import sys
sys.path.append(f'{BASE_DIR}/from_azuma')
import simulation

In [4]:
adata = sc.read_h5ad(f'{BASE_DIR}/result_32699x3000.h5ad')
adata

AnnData object with n_obs × n_vars = 32699 × 3000
    obs: 'pct_counts_mt', 'n_genes_by_counts', 'predicted_doublet', 'log1p_total_counts_hb', 'pct_counts_in_top_50_genes', 'doublet_score', 'total_counts', 'total_counts_ribo', 'pct_counts_hb', 'pct_counts_in_top_100_genes', 'n_genes', 'pct_counts_in_top_500_genes', 'pct_counts_in_top_200_genes', 'total_counts_mt', 'log1p_n_genes_by_counts', 'total_counts_hb', 'log1p_total_counts_ribo', 'cell_type', 'pct_counts_ribo', 'log1p_total_counts', 'log1p_total_counts_mt', 'origin', '_scvi_batch'
    var: 'n_cells_by_counts', 'ribo', 'pct_dropout_by_counts', 'log1p_mean_counts', 'mean_counts', 'mt', 'log1p_total_counts', 'hb', 'n_cells', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'cell_type_colors', 'hvg', 'log1p', 'neighbors', 'origin_colors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    var

In [5]:
dat = simulation.LiverCellAtlas_Simulator(sample_size=8000, method='dirichlet')

In [6]:
# 特定の項目（例：細胞型）の内訳を確認
print(adata.obs['cell_type'].value_counts())

cell_type
Hepatocytes       10625
Macrophages        7067
Neutrophils        5339
B cells            3035
CD4Tcells          2136
Monocytes          1369
Fibroblasts        1260
CD8Tcells           756
Cholangiocytes      647
NK                  334
Dendritic            70
other                61
Name: count, dtype: int64


In [7]:
# 一旦外部指定、これはあとから要改変
dat.immune_cells = ['Neutrophils', 'Macrophages', 'Monocytes', 
                    'NK', 'CD4Tcells', 'CD8Tcells', 'B cells', 'Dendritic']

In [8]:
# こっちはそのまま
dat.non_immune_cells = ['Hepatocytes', 'Cholangiocytes', 'Fibroblasts']

In [9]:
# ここもあとから要改変
dat.filter_dict = {
    'Neutrophils': 'Neutrophils', 'Monocytes & Monocyte-derived cells': 'Monocytes',
    'Kupffer cells': 'Kupffer', 'NK cells': 'NK', 'T cells': 'T', 'B cells': 'B',
    'pDCs': 'pDCs', 'cDC1s': 'cDC1s', 'cDC2s': 'cDC2s',
    'Hepatocytes': 'Hepatocytes', 'Cholangiocytes': 'Cholangiocytes', 'Fibroblasts': 'Fibroblasts',
    'Monocytes': 'Monocytes',
    'Macrophages': 'Macrophages',
    'NK': 'NK',
    'CD4Tcells': 'CD4Tcells',
    'CD8Tcells': 'CD8Tcells',
    'Dendritic': 'Dendritic'
}

In [10]:
dat.assign()
summary_df = dat.summary_df
summary_df.to_csv(f'{BASE_DIR}/data/gs_8000x11.csv')

summary_df.head()

,Neutrophils,Macrophages,Monocytes,NK,CD4Tcells,CD8Tcells,B cells,Dendritic,Hepatocytes,Cholangiocytes,Fibroblasts
0,0.028898,0.185367,0.081087,0.056220,0.010446,0.010444,0.003685,0.123854,0.048921,0.313807,0.137272
1,0.056712,0.075974,0.001283,0.216186,0.110231,0.014728,0.012383,0.012502,0.364546,0.067733,0.067721
2,0.046331,0.095014,0.072230,0.043964,0.120870,0.019188,0.044129,0.058275,0.010006,0.336309,0.153685
3,0.059017,0.149054,0.021587,0.069978,0.086984,0.004610,0.090651,0.018120,0.129452,0.002187,0.368361
4,0.003277,0.144877,0.164216,0.080501,0.017699,0.005007,0.056161,0.028262,0.401301,0.053618,0.045080


In [11]:
dat.set_data(summary_df=summary_df, cell_idx_dict=None, adata=adata)
dat.split_cell_idx(save_dir=f'{BASE_DIR}/data/cell_idx', train_ratio=0.7)
cell_idx_dict = dat.cell_idx_dict

Neutrophils: 5339 cells detected
Macrophages: 7067 cells detected
Monocytes: 1369 cells detected
NK: 334 cells detected
CD4Tcells: 2136 cells detected
CD8Tcells: 756 cells detected
B cells: 3035 cells detected
Dendritic: 70 cells detected
Hepatocytes: 10625 cells detected
Cholangiocytes: 647 cells detected
Fibroblasts: 1260 cells detected


In [12]:
pd.to_pickle(cell_idx_dict, f'{BASE_DIR}/data/cell_idx/cell_idx_dict.pkl')

In [13]:
# create train (80 samples) dataset
summary_df = pd.read_csv(f'{BASE_DIR}/data/gs_8000x11.csv', index_col=0)
cell_idx_dict = pd.read_pickle(f'{BASE_DIR}/data/cell_idx/cell_idx_dict.pkl')

summary_df = summary_df.iloc[0:10,:] # 時短で10サンプルの作成

dat = simulation.LiverCellAtlas_Simulator(sample_size=10, method='dirichlet')  
dat.set_data(summary_df=summary_df, cell_idx_dict=cell_idx_dict, adata=adata)
bulk_df = dat.create_sim_bulk(pool_size=500, mode='train')
bulk_df.to_csv(f'{BASE_DIR}/data/bulk_3000_10.csv')

bulk_df.head()

Log-transformation detected in adata.uns.


100%|██████████| 10/10 [00:04<00:00,  2.43it/s]


,0,1,2,3,4,5,6,7,8,9
7SK.2,-75.381714,443.249761,37.365547,144.550842,413.332642,-32.880352,178.942368,89.051559,49.235764,290.378375
AAGAB,359.885040,480.755440,335.612366,121.196815,556.085327,177.903488,247.016006,241.234802,448.739349,582.414382
AATK,1611.445190,146.776167,590.573853,875.622070,166.213760,385.766724,151.661438,640.876221,307.046173,710.826496
ABCA7,-25.007292,278.004089,51.193336,239.823868,0.122287,141.666702,106.473198,126.183670,199.417755,-16.019415
ABCC12,456.860352,542.365879,417.296082,450.444336,428.590240,261.348083,537.421814,327.929810,248.440613,454.051871
